In [1]:
!pip install pycocoevalcap rouge-score

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/104.3 MB ? eta -:--:--
   ---------------------------------------- 0.5/104.3 MB 4.2 MB/s eta 0:00:25
   - -------------------------------------- 3.4/104.3 MB 10.6 MB/s eta 0:00:10
   -- ------------------------------------- 6.3/104.3 MB 12.9 MB/s eta 0:00:08
   --- ------------------------------------ 8.7/104.3 MB 12.2 MB/s eta 0:00:08
   ---- ----------------------------------- 11.0/104.3 MB 11.8 MB/s eta 0:00:08
   ----- ---------------------------------- 13.6/104.3 MB 12.0 MB/s eta 0:00:08
   ------ --------------------------------- 17.0/104.3 MB 12.8 MB/s eta 0:00:07
   ------- -------------------------------- 20.2/104.3 MB 13.0 MB/s eta 0:00:07
   -------- ------------------------------- 23.1/104.3 MB 13.4 MB/s eta 0:00:07
   --------- ------------------------------ 23.6/104.3 MB 13.0 MB/s eta 0:00:07
   ---------- ------------------

In [2]:
import json
import os
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
from collections import defaultdict

# --- CẤU HÌNH ĐƯỜNG DẪN ---
PRED_FILE = "results_qwen2vl_test (2).jsonl"
GT_FILE = "data/final_dataset_v2/test/metadata.jsonl"

def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

# 1. Load dữ liệu
print("📂 Đang nạp dữ liệu...")
preds_raw = load_jsonl(PRED_FILE)
gts_raw = load_jsonl(GT_FILE)

# 2. Gom nhóm nhãn (Ground Truth): 1 ảnh có nhiều câu caption
references = defaultdict(list)
for item in gts_raw:
    img_id = item['file_name']
    references[img_id].append(item['caption'])

# 3. Chuẩn bị dữ liệu dự đoán (Prediction)
# Chỉ lấy những ảnh có xuất hiện trong cả file dự đoán và file nhãn
hypotheses = {}
for item in preds_raw:
    img_id = item['file_name']
    if img_id in references:
        hypotheses[img_id] = [item['prediction']]

# Kiểm tra số lượng
common_ids = list(hypotheses.keys())
print(f"✅ Đã khớp được {len(common_ids)} ảnh để đánh giá.")

# 4. Chuyển đổi sang định dạng yêu cầu của thư viện pycocoevalcap
# Định dạng: {image_id: [caption1, caption2, ...]}
gts = {i: references[img_id] for i, img_id in enumerate(common_ids)}
res = {i: hypotheses[img_id] for i, img_id in enumerate(common_ids)}

# 5. Khởi tạo các bộ chấm điểm
scorers = [
    (Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
    (Rouge(), "ROUGE_L"),
    (Cider(), "CIDEr")
]

# 6. Tính toán điểm số
final_results = {}
print("🧮 Đang tính toán các chỉ số...")

for scorer, method in scorers:
    score, scores = scorer.compute_score(gts, res)
    if isinstance(method, list):
        for sc, m in zip(score, method):
            final_results[m] = sc
    else:
        final_results[method] = score

# 7. In kết quả bảng đẹp
print("\n" + "="*40)
print(f"{'METRIC':<15} | {'SCORE':<10}")
print("-" * 40)
for metric, val in final_results.items():
    print(f"{metric:<15} | {val*100:.2f}" if "Bleu" in metric or "ROUGE" in metric else f"{metric:<15} | {val:.4f}")
print("="*40)
print("Lưu ý: Bleu và ROUGE được nhân với 100 để dễ quan sát.")

📂 Đang nạp dữ liệu...
✅ Đã khớp được 800 ảnh để đánh giá.
🧮 Đang tính toán các chỉ số...
{'testlen': 49280, 'reflen': 30504, 'guess': [49280, 48480, 47680, 46880], 'correct': [15372, 3567, 704, 196]}
ratio: 1.6155258326776287

METRIC          | SCORE     
----------------------------------------
Bleu_1          | 31.19
Bleu_2          | 15.15
Bleu_3          | 6.97
Bleu_4          | 3.45
ROUGE_L         | 14.94
CIDEr           | 0.0204
Lưu ý: Bleu và ROUGE được nhân với 100 để dễ quan sát.
